<div dir="rtl">

# הכנת הנתונים

המחברת הזו מיישמת בקוד את הכללים שהוכרעו בפרק המתודולוגיה. היא בונה את הטבלאות שהניתוח יעבוד עליהן, ואינה מנתחת דבר: אין כאן מבחני מובהקות, אין תרשימים ואין פרשנות. אלה נמצאים במחברת 03.

**מה המחברת כותבת.** שלושה קבצים לתיקייה `data/clean`, וכולם נוצרים מחדש בכל הרצה. אין לערוך אותם ביד, משום שההרצה הבאה תדרוס כל שינוי כזה. כל תיקון נעשה בקוד.

**מה המחברת לא נוגעת בו.** תיקיית הנתונים הגולמיים נקראת בלבד. בתחילת המחברת נרשמת טביעת אצבע של כל 11 הקבצים, והתא האחרון מוודא שהיא לא השתנתה.

**הנימוק לכל כלל** נמצא בפרק המתודולוגיה שבקובץ `book/book.md`, וסיכום מחייב שלו נמצא בסעיף ז של מחברת 01.

</div>


<div dir="rtl">

## ייבוא, קבועים וערכי הבקרה

</div>


In [1]:
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW   = Path("..") / "data" / "raw"
CLEAN = Path("..") / "data" / "clean"

K_SHRINK    = 5     # Bayesian shrinkage strength for review scores
WINDOW_DAYS = 90    # both the eligibility rule and the per-seller measurement window
MIN_DAILY_ORDERS = 10   # a day with fewer orders than this is past the end of regular data
MIN_DEALS   = 20    # a channel needs this many deals to get its own row

# Acceptance targets, computed from data/raw independently of this notebook's code.
# The pipeline must reproduce every one of them. See the final cell.
GOLDEN = {
    "data_end"               : "2018-08-29",
    "total_rows_11_tables"   : 1_559_764,
    "leads"                  : 8_000,
    "deals"                  : 842,
    "deals_eligible"         : 665,
    "activated_in_window"    : 292,
    "activation_pct"         : 43.9,
    "paid_search_activated"  : 81,
    "paid_search_deals"      : 146,
    "sellers_ever_sold"      : 380,
    "sellers_scored"         : 341,
    "sellers_with_channel"   : 337,
    "delivered_item_rows"    : 4_654,
    "revenue_before_window"  : 617_977,
    "repeated_order_product" : 7_088,
    "repeated_in_revenue_grain": 287,
    "no_activity_in_window"  : 52,
    "zero_span_sellers"      : 69,
    "zero_span_one_order"    : 68,
    "largest_channel_sellers": 102,
}
results = {}   # filled in as we go, checked at the end

print("pandas", pd.__version__, "| numpy", np.__version__)


pandas 3.0.5 | numpy 2.4.6


<div dir="rtl">

## טעינת הטבלאות וטביעת אצבע של הנתונים הגולמיים

נטענות כל 11 הטבלאות, כדי לאמת שסך השורות תואם את מה שמתועד במאגר. חמש מהן אינן משמשות את הצנרת הזו כלל: הגאולוקציה, המוצרים, הלקוחות, התשלומים ותרגום הקטגוריות, שנטענות כאן בשמות `geolocation`, `products`, `customers`, `payments` ו-`categories` בהתאמה. הן נטענות לבדיקת שלמות בלבד, וסימונן המפורש עדיף על השמטה שקטה.

הטבלה `sellers` נטענת אף היא, אך לא כתלות. היא משמשת כבדיקה: היא מכילה אך ורק מוכרים שמכרו בפועל, וזו הסיבה שהיחס בין 380 ל-842 הוא שיעור המוכרים שמכרו אי פעם ולא נתון חסר.

</div>


In [2]:
FILES = {
    "deals"      : "olist_closed_deals_dataset.csv",
    "leads"      : "olist_marketing_qualified_leads_dataset.csv",
    "sellers"    : "olist_sellers_dataset.csv",
    "items"      : "olist_order_items_dataset.csv",
    "orders"     : "olist_orders_dataset.csv",
    "reviews"    : "olist_order_reviews_dataset.csv",
    "customers"  : "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "payments"   : "olist_order_payments_dataset.csv",
    "products"   : "olist_products_dataset.csv",
    "categories" : "product_category_name_translation.csv",
}
DATES = {
    "deals"  : ["won_date"],
    "leads"  : ["first_contact_date"],
    "orders" : ["order_purchase_timestamp", "order_approved_at",
                "order_delivered_carrier_date", "order_delivered_customer_date",
                "order_estimated_delivery_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}

def raw_fingerprint():
    """sha256 over every raw CSV, so we can prove data/raw was never written to."""
    h = hashlib.sha256()
    for name in sorted(FILES.values()):
        h.update(hashlib.sha256((RAW / name).read_bytes()).hexdigest().encode())
    return h.hexdigest()

FINGERPRINT_BEFORE = raw_fingerprint()

tables = {k: pd.read_csv(RAW / v, parse_dates=DATES.get(k)) for k, v in FILES.items()}
total  = sum(len(t) for t in tables.values())

for k in FILES:
    print("%-12s %9d rows" % (k, len(tables[k])))
print("-" * 28)
print("%-12s %9d rows" % ("TOTAL", total))
assert total == GOLDEN["total_rows_11_tables"], "raw row count changed"

results["total_rows_11_tables"] = total

# the five the pipeline actually uses
deals, leads   = tables["deals"], tables["leads"]
items, orders  = tables["items"], tables["orders"]
reviews        = tables["reviews"]

# sixth, loaded as a check rather than a dependency
assert set(tables["sellers"].seller_id) == set(items.seller_id), \
    "the sellers table is meant to be exactly the set of sellers that transacted"
results["leads"] = len(leads)
print("\nsellers table == sellers appearing in order_items:", len(tables["sellers"]), "ids")
print("raw fingerprint:", FINGERPRINT_BEFORE[:16], "...")


deals              842 rows
leads             8000 rows
sellers           3095 rows
items           112650 rows
orders           99441 rows
reviews          99224 rows
customers        99441 rows
geolocation    1000163 rows
payments        103886 rows
products         32951 rows
categories          71 rows
----------------------------
TOTAL          1559764 rows

sellers table == sellers appearing in order_items: 3095 ids
raw fingerprint: 42fca2bccb836569 ...


<div dir="rtl">

## בניית אוכלוסיית המחקר

עמודת המקור `origin` קיימת אך ורק בטבלת הלידים, ולכן החיבור אליה אינו אופציונלי. בלעדיו אין לאף עסקה ערוץ.

**אין למלא ערכים ריקים בעמודה הזו.** ערך ריק ו-`unknown` הם שני דברים שונים: `unknown` היא קטגוריה אמיתית בנתוני המקור, וערך ריק הוא היעדר רישום. ההבחנה הזו היא שמפרידה בין 341 המוכרים שקיבלו ציון לבין 337 בעלי ערוץ מזוהה.

</div>


In [3]:
deals_o = deals.merge(leads[["mql_id", "origin"]], on="mql_id", how="left", validate="one_to_one")
assert len(deals_o) == GOLDEN["deals"], "the origin join changed the row count"

results["deals"] = len(deals_o)
print("deals                 :", len(deals_o))
print("origin is null        :", int(deals_o.origin.isna().sum()), "(absence of a record)")
print("origin == 'unknown'   :", int((deals_o.origin == "unknown").sum()), "(a real source category)")
print()
print(deals_o.origin.value_counts(dropna=False).to_string())


deals                 : 842
origin is null        : 14 (absence of a record)
origin == 'unknown'   : 179 (a real source category)

origin
organic_search       271
paid_search          195
unknown              179
social                75
direct_traffic        56
referral              24
email                 15
NaN                   14
display                6
other                  4
other_publicities      3


<div dir="rtl">

## שני כללי תשעים הימים

המספר תשעים ממלא כאן שני תפקידים שונים, וערבוב ביניהם הוא הטעות הקלה ביותר לעשות במחברת הזו:

* **כשירות עסקה.** עסקה נכללת רק אם חלפו לפחות 90 ימים קלנדריים בין מועד החתימה שלה לבין סוף נתוני ההזמנות. הכלל מותיר 665 עסקאות מתוך 842.
* **חלון מדידה לכל מוכר.** לכל עסקה נספרים 90 ימים קלנדריים ממועד החתימה שלה עצמה, והיום התשעים נכלל בפנים.

הראשון קובע מי נכנס להשוואה, והשני קובע מה נמדד עליו. הסיבה לשניהם אחת: נתוני ההזמנות נגמרים לפני שעסקאות מאוחרות הספיקו לצבור תשעים יום, ולכן מוכר שחתם מאוחר לא קיבל הזדמנות שווה.

**היכן נגמרים נתוני ההזמנות.** ההזמנה האחרונה בטבלה רשומה ב-17 באוקטובר 2018, אבל זה אינו סוף הנתונים בפועל. עד אמצע אוגוסט נרשמות מאות הזמנות ביום, ב-29 באוגוסט ארבע עשרה, ואחריו נרשמות עשרים וחמש הזמנות בלבד בשבעה שבועות, ועשרים וארבע מהן בוטלו. חלון שנמשך אל תוך התקופה הזו אינו נצפה באמת, ולכן סוף הנתונים מוגדר כיום האחרון שבו נרשמו לפחות עשר הזמנות: 29 באוגוסט 2018. זהו גם היום שבו נרשמה ההזמנה האחרונה של מוכר כלשהו בקבוצת המחקר.

</div>


In [4]:
# The order table runs to 2018-10-17 only because of a few stray orders. Regular
# data ends at the last day with at least MIN_DAILY_ORDERS orders; a window that
# runs past it cannot be observed.
daily_orders = orders.order_purchase_timestamp.dt.normalize().value_counts().sort_index()
DATA_END = daily_orders[daily_orders >= MIN_DAILY_ORDERS].index.max()
after_end = orders[orders.order_purchase_timestamp.dt.normalize() > DATA_END]
print("last order in the table  :", orders.order_purchase_timestamp.max().date())
print("orders per day, last week:", daily_orders[DATA_END - pd.Timedelta(days=6):DATA_END].tolist())
print("orders after the end     : %d, of them canceled %d" % (len(after_end), (after_end.order_status == "canceled").sum()))
results["data_end"] = str(DATA_END.date())
assert results["data_end"] == GOLDEN["data_end"]

deals_o["won_day"]       = deals_o.won_date.dt.normalize()
deals_o["exposure_days"] = (DATA_END - deals_o.won_day).dt.days
deals_o["eligible"]      = deals_o.exposure_days >= WINDOW_DAYS      # rule 1
deals_o["window_end"]    = deals_o.won_day + pd.Timedelta(days=WINDOW_DAYS)  # rule 2

results["deals_eligible"] = int(deals_o.eligible.sum())
print("order data ends        :", DATA_END.date())
print("deals eligible         : %d of %d" % (deals_o.eligible.sum(), len(deals_o)))
print("eligible signing range : %s .. %s"
      % (deals_o.loc[deals_o.eligible, "won_day"].min().date(),
         deals_o.loc[deals_o.eligible, "won_day"].max().date()))
assert results["deals_eligible"] == GOLDEN["deals_eligible"]


last order in the table  : 2018-10-17
orders per day, last week: [144, 99, 69, 73, 67, 44, 14]
orders after the end     : 25, of them canceled 24
order data ends        : 2018-08-29
deals eligible         : 665 of 842
eligible signing range : 2017-12-05 .. 2018-05-31


<div dir="rtl">

## שתי מסגרות הזמנות, לא אחת

הסינון לסטטוס שסופק יוצר מסגרת נתונים **שנייה** ואינו מחליף את טבלת ההזמנות המלאה.

**הפעלה נמדדת על הזמנות מכל סטטוס**, וזהו המקום היחיד בפרויקט שאינו מסונן. הסיבה: הפעלה שואלת האם המוכר התחיל למכור, והזמנה שבוטלה היא עדיין תחילת מכירה. **הכנסות, ביקורות ומשך פעילות** נמדדים על הזמנות שסופקו בלבד, משום שהזמנה שלא סופקה מקבלת ציון ביקורת נמוך בהרבה: בכלל הנתונים הממוצע הוא 4.16 בהזמנות שסופקו, ובין 1.28 ל-2.01 בכל סטטוס אחר שיש בו יותר ממאה ביקורות, כפי שמראה מחברת 01. ספירתה הייתה מודדת כשל לוגיסטי כאי שביעות רצון.

</div>


In [5]:
orders_all       = orders                                              # activation only
orders_delivered = orders[orders.order_status == "delivered"].copy()    # revenue and reviews

items_all       = items.merge(orders_all[["order_id", "order_purchase_timestamp"]],
                              on="order_id", how="inner")
items_delivered = items.merge(orders_delivered[["order_id", "order_purchase_timestamp"]],
                              on="order_id", how="inner")

print("orders, all statuses :", len(orders_all))
print("orders, delivered    :", len(orders_delivered))
print("item rows, all       :", len(items_all))
print("item rows, delivered :", len(items_delivered))

# repeated order+product rows are quantity, never duplication: they must survive
repeated = int((items.groupby(["order_id", "product_id"]).size() > 1).sum())
results["repeated_order_product"] = repeated
print("\nrepeated order+product combinations:", repeated, "(must not be removed)")
assert repeated == GOLDEN["repeated_order_product"]


orders, all statuses : 99441
orders, delivered    : 96478
item rows, all       : 112650
item rows, delivered : 110197

repeated order+product combinations: 7088 (must not be removed)


<div dir="rtl">

## הפעלה

מוכר נחשב מופעל אם ביצע הזמנה ראשונה בתוך 90 ימים קלנדריים ממועד חתימת העסקה שלו, **בכל סטטוס**.

שימו לב להבחנה שמופיעה כאן במספרים: מתוך 665 העסקאות הכשירות, 345 הגיעו למכירה אי פעם, אך רק 292 מכרו בתוך החלון. 53 העסקאות שביניהן מכרו מאוחר יותר, בחציון של 113 ימים. הן נספרות כלא מופעלות, ולכן השיעור המדווח מודד **מהירות** עד המכירה הראשונה ולא עצם קיומה.

</div>


In [6]:
first_sale = (items_all.groupby("seller_id").order_purchase_timestamp.min().dt.normalize())

deals_o["first_sale_date"] = deals_o.seller_id.map(first_sale)
deals_o["days_to_first_sale"] = (deals_o.first_sale_date - deals_o.won_day).dt.days
# The assert rules out a first sale before signing; the lower bound stays in the
# definition so that it reads complete without the assert.
assert deals_o.days_to_first_sale.dropna().min() >= 0, "a seller sold before signing"
deals_o["activated"] = (deals_o.first_sale_date.notna()
                        & (deals_o.days_to_first_sale >= 0)
                        & (deals_o.days_to_first_sale <= WINDOW_DAYS))

elig = deals_o[deals_o.eligible]
results["activated_in_window"] = int(elig.activated.sum())
results["activation_pct"]      = round(100 * elig.activated.mean(), 1)
results["sellers_ever_sold"]   = int(deals_o.first_sale_date.notna().sum())

ps = elig[elig.origin == "paid_search"]
results["paid_search_activated"] = int(ps.activated.sum())
results["paid_search_deals"]     = int(len(ps))

print("activated within window : %d of %d  (%.1f%%)"
      % (results["activated_in_window"], len(elig), results["activation_pct"]))
print("ever sold, eligible     : %d" % elig.first_sale_date.notna().sum())
print("sold after day 90       : %d  (median %d days)"
      % ((elig.days_to_first_sale > WINDOW_DAYS).sum(),
         elig.loc[elig.days_to_first_sale > WINDOW_DAYS, "days_to_first_sale"].median()))
print("paid_search             : %d of %d" % (results["paid_search_activated"], results["paid_search_deals"]))
print("ever sold, all 842      : %d" % results["sellers_ever_sold"])

# If activation were accidentally filtered to delivered orders these two would fall.
assert results["activated_in_window"]   == GOLDEN["activated_in_window"]
assert results["paid_search_activated"] == GOLDEN["paid_search_activated"]


activated within window : 292 of 665  (43.9%)
ever sold, eligible     : 345
sold after day 90       : 53  (median 113 days)
paid_search             : 81 of 146
ever sold, all 842      : 380


<div dir="rtl">

## הקובץ הראשון: טבלת העסקאות

הקובץ נכתב עבור כל 842 העסקאות, וכשירות לחלון נשמרת כדגל ולא כסינון. כך מחברת 03 תוכל להראות את השפעת הקטימה בלי לחשב מחדש.

</div>


In [7]:
CLEAN.mkdir(parents=True, exist_ok=True)

FOOTNOTE = "other_publicities"          # shown separately, never merged
MERGED   = "small_channels_merged"

deal_counts = deals_o.origin.value_counts()          # threshold measured on all 842
big   = set(deal_counts[deal_counts >= MIN_DEALS].index)
small = sorted(set(deal_counts.index) - big - {FOOTNOTE})

def bucket(o):
    """Channel label for reporting. Empty origin belongs to no channel."""
    if pd.isna(o):    return np.nan
    if o in big:      return o
    if o == FOOTNOTE: return FOOTNOTE
    return MERGED

# Assigned here, not later, so the grouping rule is persisted in every output file
# and notebook 03 never has to reimplement it.
deals_o["channel"] = deals_o.origin.map(bucket)
print("own row  :", sorted(big))
print("merged   :", small)
print("footnote :", [FOOTNOTE])

final_deals = deals_o[[
    "mql_id", "seller_id", "origin", "channel", "won_date", "won_day",
    "exposure_days", "eligible", "first_sale_date", "days_to_first_sale", "activated",
]].copy()

final_deals.to_csv(CLEAN / "final_deals.csv", index=False)
print("wrote final_deals.csv :", final_deals.shape)
display(final_deals.head())


own row  : ['direct_traffic', 'organic_search', 'paid_search', 'referral', 'social', 'unknown']
merged   : ['display', 'email', 'other']
footnote : ['other_publicities']
wrote final_deals.csv : (842, 11)


,mql_id,seller_id,origin,channel,won_date,won_day,exposure_days,eligible,first_sale_date,days_to_first_sale,activated
0,5420aad7fec3549a85876ba1c529bd84,2c43fb513632d29b3b58df74816f1b06,organic_search,organic_search,2018-02-26 19:58:54,2018-02-26,184,True,2018-06-20,114.0,False
1,a555fb36b9368110ede0f043dfc3b9a0,bbb7d7893a450660432ea6652310ebb7,referral,referral,2018-05-08 20:17:59,2018-05-08,113,True,NaT,NaN,False
2,327174d3648a2d047e8940d7d15204ca,612170e34b97004b3ba37eae81836b4c,organic_search,organic_search,2018-06-05 17:27:23,2018-06-05,85,False,2018-06-11,6.0,True
3,f5fee8f7da74f4887f5bcae2bafb6dd6,21e1781e36faf92725dde4730a88ca0f,paid_search,paid_search,2018-01-17 13:51:03,2018-01-17,224,True,NaT,NaN,False
4,ffe640179b554e295c167a2f6be528e0,ed8cb7b190ceb6067227478e48cf8dde,unknown,unknown,2018-07-03 20:17:45,2018-07-03,57,False,2018-07-19,16.0,True


<div dir="rtl">

## הקובץ השני: טבלת הערוצים, גרסה ראשונה

ערוץ עם 20 עסקאות ומעלה מקבל שורה נפרדת, והשאר מאוחדים לשורה אחת שמצוין מה היא כוללת. הסף נמדד על כל 842 העסקאות.

**חריג אחד:** הערוץ `other_publicities` מוצג בנפרד ולא מאוחד. יש לו 65 לידים, שלוש עסקאות ואפס מוכרים. זהו אפס אמיתי וממצא בפני עצמו, ואיחודו לתוך קבוצה היה מטשטש אותו.

רווחי הסמך מחושבים בשיטת וילסון ולא בקירוב הנורמלי, משום שחלק מהערוצים קטנים ושיעוריהם קרובים לקצה, ובתנאים כאלה הקירוב הנורמלי מייצר גבולות מחוץ לתחום.

</div>


In [8]:
def wilson(k, n, z=1.96):
    """Wilson score interval: behaves at small n and near 0 or 1, unlike the normal approximation."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (max(0.0, c - h), min(1.0, c + h))

leads_ch = leads.assign(channel=leads.origin.map(bucket))

# Column names say which population each count is over, so a reader of the CSV
# cannot accidentally build a rate from the uncorrected denominator.
ch = (deals_o.groupby("channel")
      .agg(deals_all=("mql_id", "size"),
           activated_all_deals=("activated", "sum"),
           sellers_ever_sold=("first_sale_date", "count"))
      .join(leads_ch.groupby("channel").size().rename("leads"))
      .reset_index())

elig_ch = (deals_o[deals_o.eligible].groupby("channel")
           .agg(deals_eligible=("eligible", "size"), activated_eligible=("activated", "sum")))
ch = ch.merge(elig_ch, on="channel", how="left")

# The only rate the project reports: activated within window, over eligible deals.
ch["conversion_rate"] = ch.deals_all / ch.leads
ch["activation_rate"] = ch.activated_eligible / ch.deals_eligible
ch[["activation_ci_low", "activation_ci_high"]] = ch.apply(
    lambda r: pd.Series(wilson(r.activated_eligible, r.deals_eligible)), axis=1)

# Carried into the file so the rule travels with the data, not just the prose.
ch["excluded_from_ranking"] = ch.channel.isin(["unknown", FOOTNOTE, MERGED])

ch = ch.sort_values("activation_rate", ascending=False, na_position="last")
display(ch)


,channel,deals_all,activated_all_deals,sellers_ever_sold,leads,deals_eligible,activated_eligible,conversion_rate,activation_rate,activation_ci_low,activation_ci_high,excluded_from_ranking
3,paid_search,195,94,101,1586,146,81,0.122951,0.554795,0.473801,0.632979,False
0,direct_traffic,56,25,31,499,49,22,0.112224,0.448980,0.318526,0.586851,False
4,referral,24,8,9,284,18,8,0.084507,0.444444,0.245592,0.662839,False
6,social,75,29,31,1350,55,24,0.055556,0.436364,0.313732,0.567304,False
7,unknown,179,68,81,1099,153,63,0.162875,0.411765,0.336872,0.490980,True
1,organic_search,271,93,113,2296,207,84,0.118031,0.405797,0.341209,0.473818,False
5,small_channels_merged,25,7,10,761,23,7,0.032852,0.304348,0.156038,0.508661,True
2,other_publicities,3,0,0,65,3,0,0.046154,0.000000,0.000000,0.561506,True


<div dir="rtl">

# מעבר שני: ציון ערך המוכר

עד כאן נבנה הממצא: שיעורי הפעלה לפי ערוץ, עם רווחי סמך. מכאן ואילך נבנה המדד שמשמש לבדיקת החלק השני של שאלת המחקר, זה שעוסק באיכות המוכרים.

## צמצום טבלת הביקורות לשורה אחת לכל הזמנה

**חובה לצמצם לפני כל חיבור.** בלי הצמצום הזה, הזמנה עם שתי ביקורות מכפילה את שורות הפריט שלה בחיבור, וההכנסות נספרות פעמיים.

**שימו לב:** הפונקציה `drop_duplicates` לבדה אינה פותרת את זה, משום שאין בטבלה שורות כפולות מלאות. השורות נבדלות בתוכן הביקורת, ורק מזהה ההזמנה חוזר.

הכלל שנקבע: הביקורת הראשונה לפי תאריך, ושבירת שוויון לפי שעת המענה. שבירת השוויון נחוצה משום שעמודת התאריך מכילה יום בלבד, ו-155 הזמנות נושאות שתי ביקורות באותו יום.

</div>


In [9]:
dupes = reviews.groupby("order_id").size()
print("review rows                :", len(reviews))
print("unique order_id            :", reviews.order_id.nunique())
print("surplus rows               :", len(reviews) - reviews.order_id.nunique())
print("orders with >1 review      :", int((dupes > 1).sum()))

multi = reviews[reviews.order_id.isin(dupes[dupes > 1].index)]
same_day = multi.groupby("order_id").review_creation_date.nunique()
print("of those, same-day reviews :", int((same_day == 1).sum()), "-> the answer-time tiebreak matters")

# head(1) keeps a real review row. groupby(...).first() would take the first
# non-null value of each column independently, producing a composite row that
# matches no actual review: harmless for review_score, wrong as a record.
reviews_one = (reviews
               .sort_values(["order_id", "review_creation_date", "review_answer_timestamp"],
                            kind="mergesort")
               .groupby("order_id", as_index=False)
               .head(1))

assert len(reviews_one) == reviews.order_id.nunique()
assert reviews_one.order_id.is_unique
# NOT asserted: review_id uniqueness. One review legitimately covers several orders
# of the same purchase: 789 ids across 1,603 rows, recorded in notebook 01.

# The check that can actually fail: the kept review must be the earliest of its order.
earliest = reviews.groupby("order_id").review_creation_date.min().rename("earliest")
chk = reviews_one.merge(earliest, on="order_id")
assert (chk.review_creation_date == chk.earliest).all(), "collapse did not keep the earliest review"
print("\nafter collapse             :", len(reviews_one), "rows, one per order")


review rows                : 99224
unique order_id            : 98673
surplus rows               : 551
orders with >1 review      : 547
of those, same-day reviews : 155 -> the answer-time tiebreak matters



after collapse             : 98673 rows, one per order


<div dir="rtl">

## אוכלוסיית הניקוד

הציון נמדד רק על מוכרים מעסקאות כשירות, כדי שחלון התשעים יום של כל אחד מהם נצפה במלואו. מתוכם, 345 מוכרים ביצעו הזמנה כלשהי. רכיבי ההכנסות ושביעות הרצון נמדדים על הזמנות שסופקו, ולכן מוכר שאף הזמנה שלו לא סופקה אינו ניתן לניקוד, ואוכלוסיית הציון היא 341 מוכרים.

מתוך 341 המוכרים שקיבלו ציון, ארבעה מגיעים מלידים שעמודת המקור שלהם ריקה, ולכן מספר המוכרים להשוואה בין ערוצים הוא 337.

</div>


In [10]:
sellers_delivered = set(items_delivered.seller_id)
eligible_sellers  = set(deals_o.loc[deals_o.eligible, "seller_id"].dropna())
scored_ids = sorted(eligible_sellers & sellers_delivered)

results["sellers_scored"] = len(scored_ids)
sd = deals_o[deals_o.seller_id.isin(scored_ids)]
results["sellers_with_channel"] = int(sd.origin.notna().sum())

ever_sold_eligible = len(eligible_sellers & set(items_all.seller_id))
print("sellers that ever ordered, all deals      :", results["sellers_ever_sold"])
print("  of those, from eligible deals          :", ever_sold_eligible)
print("  of those, with a delivered order       :", results["sellers_scored"], "  <- scored population")
print("  ordered but never had a delivery       :", ever_sold_eligible - results["sellers_scored"])
print()
print("scored sellers with an identified channel:", results["sellers_with_channel"])
print("  difference                             :",
      results["sellers_scored"] - results["sellers_with_channel"], "come from empty-origin leads")

assert results["sellers_scored"]       == GOLDEN["sellers_scored"]
assert results["sellers_with_channel"] == GOLDEN["sellers_with_channel"]

# Every scored seller comes from an eligible deal, so every measurement window was observed in full.
exposure_scored = deals_o.loc[deals_o.seller_id.isin(scored_ids), "exposure_days"]
print("\nminimum exposure_days among scored sellers:", int(exposure_scored.min()))
assert (exposure_scored >= WINDOW_DAYS).all(), "a scored seller's window runs past the end of the data"


sellers that ever ordered, all deals      : 380
  of those, from eligible deals          : 345
  of those, with a delivered order       : 341   <- scored population
  ordered but never had a delivery       : 4

scored sellers with an identified channel: 337
  difference                             : 4 come from empty-origin leads

minimum exposure_days among scored sellers: 90


<div dir="rtl">

## הכנסות ושביעות רצון בתוך חלון המדידה

**הכנסות מגיעות מעמודת `price` בטבלת הפריטים בלבד**, ולעולם לא מסכום התשלום. סכום התשלום כולל דמי משלוח ומחולק לפי אמצעי תשלום, ואינו מייצג את הכנסת המוכר.

כל שורת פריט משויכת לחלון של המוכר שלה: היא נספרת רק אם מועד הרכישה נופל בין יום החתימה ליום התשעים שאחריו, כולל.

**לפי איזה תאריך ביקורת נכנסת לחלון.** ביקורת נכנסת לחלון לפי מועד הרכישה של ההזמנה שלה, ולא לפי מועד יצירת הביקורת. הנימוק הוא שההכנסות והביקורות יימדדו על אותו ציר זמן, ושתיהן מתייחסות לאותה פעילות מכירה. ההכרעה השנייה הייתה נותנת 1,774 ביקורות במקום 2,227, ומזיזה את הממוצע הכללי מ-4.229 ל-4.242 בלבד. הבחירה אינה משנה את מספר המוכרים חסרי הפעילות בחלון, שנותר 52 בשתי הקריאות.

</div>


In [11]:
# One deal per seller across all 842, so no seller carries two competing windows.
# Asserted rather than assumed: drop_duplicates would otherwise pick one silently.
assert deals_o.seller_id.dropna().is_unique, "a seller appears on more than one deal"

win = (deals_o[deals_o.seller_id.isin(scored_ids)]
       [["seller_id", "channel", "won_day", "window_end", "exposure_days", "origin"]])

sv = items_delivered[items_delivered.seller_id.isin(scored_ids)].merge(
        win, on="seller_id", how="inner", validate="many_to_one")
results["delivered_item_rows"]   = len(sv)
results["revenue_before_window"] = int(round(sv.price.sum()))

# The GOLDEN["repeated_order_product"] check in the loading cell guards the raw input,
# which nothing mutates, so it cannot detect a dedup introduced here. This one can:
# it counts repeated rows on the exact grain that revenue is summed from.
results["repeated_in_revenue_grain"] = int((sv.groupby(["order_id", "product_id"]).size() > 1).sum())
print("repeated order+product rows inside the revenue grain:", results["repeated_in_revenue_grain"])
assert results["repeated_in_revenue_grain"] == GOLDEN["repeated_in_revenue_grain"], (
    "repeated item rows were removed: they are quantity, and revenue is now understated")
print("delivered item rows, before window :", results["delivered_item_rows"])
print("revenue, before window             : %d R$" % results["revenue_before_window"])
assert results["delivered_item_rows"]   == GOLDEN["delivered_item_rows"]
assert results["revenue_before_window"] == GOLDEN["revenue_before_window"]

day = sv.order_purchase_timestamp.dt.normalize()
sv["in_window"] = (day >= sv.won_day) & (day <= sv.window_end)
inw = sv[sv.in_window]
print("\ndelivered item rows, inside window :", len(inw))
print("revenue, inside window             : %d R$" % round(inw.price.sum()))

revenue_w = inw.groupby("seller_id").price.sum()

# one review per order, joined only to orders inside the window
order_seller = inw[["seller_id", "order_id"]].drop_duplicates()
rv = order_seller.merge(reviews_one[["order_id", "review_score"]], on="order_id", how="inner",
                        validate="many_to_one")
review_mean_w = rv.groupby("seller_id").review_score.mean()
review_n_w    = rv.groupby("seller_id").review_score.size()

no_activity = [s for s in scored_ids if s not in revenue_w.index and s not in review_n_w.index]
results["no_activity_in_window"] = len(no_activity)
print("\nsellers with no revenue and no review inside the window:", len(no_activity))
assert results["no_activity_in_window"] == GOLDEN["no_activity_in_window"]

# The alternative weighed above: a review enters the window by the date it was written,
# not by the purchase date of its order. Computed only to report the difference.
alt = (sv[["seller_id", "order_id", "won_day", "window_end"]].drop_duplicates()
       .merge(reviews_one[["order_id", "review_score", "review_creation_date"]], on="order_id",
              how="inner", validate="many_to_one"))
alt = alt[(alt.review_creation_date.dt.normalize() >= alt.won_day)
          & (alt.review_creation_date.dt.normalize() <= alt.window_end)]
alt_none = [s for s in scored_ids if s not in revenue_w.index and s not in set(alt.seller_id)]
print("\nreviews in window, by purchase date : %d, mean %.3f" % (len(rv), rv.review_score.mean()))
print("reviews in window, by review date   : %d, mean %.3f, sellers with no activity: %d"
      % (len(alt), alt.review_score.mean(), len(alt_none)))


repeated order+product rows inside the revenue grain: 287
delivered item rows, before window : 4654
revenue, before window             : 617977 R$

delivered item rows, inside window : 2482
revenue, inside window             : 336708 R$

sellers with no revenue and no review inside the window: 52

reviews in window, by purchase date : 2227, mean 4.229
reviews in window, by review date   : 1774, mean 4.242, sellers with no activity: 52


<div dir="rtl">

## שלושת רכיבי הציון

**הכנסות בלוגריתם בסיס עשר.** בקנה מידה ליניארי, מחצית המוכרים המרכזית נדחסת לנקודה אחת מתוך מאה, משום שמוכר בודד אחראי לכחמישית מהכנסות הקבוצה. הלוגריתם פורש את ההתפלגות.

**הממוצע שאליו מכווצים.** הממוצע הכללי מחושב על פני כל הביקורות שבתוך החלון, ולא כממוצע של ממוצעי המוכרים. המשמעות היא שמוכר עם ביקורות רבות משפיע עליו יותר. זו בחירה מודעת: היעד הוא ציון הביקורת הטיפוסי בקבוצה, ולא הציון של המוכר הטיפוסי.

**ביקורות בכיווץ בייסיאני, עם `k=5` כעוצמת הכיווץ.** חציון הביקורות בתוך החלון הוא שתיים בלבד, ו-65 מוכרים היו מקבלים ציון מושלם על סמך ביקורת אחת. הנוסחה מושכת מוכרים עם מעט ביקורות אל הממוצע הכללי. עבור מוכר בלי ביקורות כלל, `n=0`, והנוסחה מחזירה בדיוק את הממוצע הכללי.

**משך פעילות, ובמפורש ללא הגבלת חלון.** החלת החלון גם על הרכיב הזה הייתה סותרת את החלטת היקף המדידה: משך פעילות שנחתך ב-90 יום אינו מודד עמידות אלא את החלון עצמו.

**המכנה הוא הימים מההזמנה הראשונה של המוכר ועד סוף נתוני ההזמנות.** אין להשתמש כאן בימים שמאז חתימת העסקה. שני הגדלים נשמעים דומים, אך השני שולט רק בכשירות העסקה, והחלפה ביניהם משנה את הציון של כל מוכר. לכן הם מוחזקים בשתי עמודות בעלות שמות נפרדים, `exposure_days` ו-`tenure_exposure_days`, כדי שאי אפשר יהיה להחליף ביניהם בטעות.

</div>


In [12]:
scale = lambda s: (s - s.min()) / (s.max() - s.min()) if s.max() > s.min() else s * 0

sc = pd.DataFrame({"seller_id": scored_ids}).merge(win, on="seller_id", how="left")

# 1. revenue, inside the window, log base 10.
# All-time revenue is carried alongside it because the book's output spec asks for
# both: the score uses the windowed figure, the report shows what the window left out.
revenue_all = sv.groupby("seller_id").price.sum()
sc["revenue_window"]         = sc.seller_id.map(revenue_w).fillna(0.0)
sc["revenue_all_time"]       = sc.seller_id.map(revenue_all).fillna(0.0)
sc["revenue_outside_window"] = sc.revenue_all_time - sc.revenue_window
assert (sc.revenue_outside_window >= -1e-6).all(), "windowed revenue exceeds all-time revenue"
assert int(round(sc.revenue_all_time.sum())) == GOLDEN["revenue_before_window"], (
    "all-time revenue must reconcile with the pre-window total")
sc["revenue_log"]    = np.log10(sc.revenue_window + 1)

# 2. satisfaction, shrunk toward the group mean
GLOBAL_MEAN = rv.review_score.mean()
sc["reviews_n"]    = sc.seller_id.map(review_n_w).fillna(0).astype(int)
sc["review_mean"]  = sc.seller_id.map(review_mean_w)
sc["review_shrunk"] = ((sc.reviews_n * sc.review_mean.fillna(0) + K_SHRINK * GLOBAL_MEAN)
                       / (sc.reviews_n + K_SHRINK))
print("sellers whose only review in the window is five stars:", int(((sc.reviews_n == 1) & (sc.review_mean == 5)).sum()))

# 3. staying power, normalised by exposure, not windowed
span = (items_delivered[items_delivered.seller_id.isin(scored_ids)]
        .groupby("seller_id").order_purchase_timestamp
        .agg(lambda s: (s.max().normalize() - s.min().normalize()).days))
n_orders = (items_delivered[items_delivered.seller_id.isin(scored_ids)]
            .groupby("seller_id").order_id.nunique())
first_order = (items_delivered[items_delivered.seller_id.isin(scored_ids)]
                .groupby("seller_id").order_purchase_timestamp.min().dt.normalize())

sc["active_span_days"] = sc.seller_id.map(span).fillna(0).astype(int)
sc["n_orders"]         = sc.seller_id.map(n_orders).fillna(0).astype(int)

# The denominator is days from the seller's FIRST ORDER to the end of the order
# data, as defined in the methodology chapter, not days since signing. The two
# are different quantities: `exposure_days` (signing-based) decides which deals
# are eligible; this one asks what share of a seller's selling life was active.
sc["first_order_day"]       = sc.seller_id.map(first_order)
sc["tenure_exposure_days"]  = (DATA_END - sc.first_order_day).dt.days
sc["tenure_ratio"]          = sc.active_span_days / sc.tenure_exposure_days

assert (sc.tenure_exposure_days > 0).all(), "a scored seller has no positive selling window"
assert sc.tenure_ratio.between(0, 1).all(), "active span cannot exceed the selling window"
last_cohort_order = items_delivered[items_delivered.seller_id.isin(scored_ids)].order_purchase_timestamp.max().normalize()
assert last_cohort_order <= DATA_END, "a scored seller has an order after the end of the data"
print("highest tenure ratio:", round(sc.tenure_ratio.max(), 3), "| last cohort order:", last_cohort_order.date())

results["zero_span_sellers"]   = int((sc.active_span_days == 0).sum())
results["zero_span_one_order"] = int(((sc.active_span_days == 0) & (sc.n_orders == 1)).sum())
print("global mean review score (in-window):", round(GLOBAL_MEAN, 3))
print("median reviews per seller           :", int(sc.reviews_n.median()))
print("zero-span sellers                   :", results["zero_span_sellers"],
      "| of those with a single order:", results["zero_span_one_order"])
assert results["zero_span_sellers"]   == GOLDEN["zero_span_sellers"]
assert results["zero_span_one_order"] == GOLDEN["zero_span_one_order"]

sc["revenue_component"]      = scale(sc.revenue_log)
sc["satisfaction_component"] = scale(sc.review_shrunk)
sc["tenure_component"]       = scale(sc.tenure_ratio)
display(sc[["revenue_component", "satisfaction_component", "tenure_component"]].describe().round(3))


sellers whose only review in the window is five stars: 65
highest tenure ratio: 1.0 | last cohort order: 2018-08-29
global mean review score (in-window): 4.229
median reviews per seller           : 2
zero-span sellers                   : 69 | of those with a single order: 68


,revenue_component,satisfaction_component,tenure_component
count,341.000,341.000,341.000
mean,0.440,0.690,0.608
std,0.221,0.160,0.368
min,0.000,0.000,0.000
25%,0.360,0.644,0.246
50%,0.481,0.718,0.773
75%,0.589,0.768,0.911
max,1.000,1.000,1.000


<div dir="rtl">

## הקובץ השלישי: ציון ערך המוכר

המשקלות שווים, וטבלת הרגישות שבודקת עד כמה הדירוג תלוי בבחירה הזו נבנית במחברת 03. הציון מוצג בסולם 1 עד 100.

**הציון הוא כלי לדירוג מוכרים בודדים ולא בסיס לדירוג ערוצים.** מבחן התמורות שבמחברת הניתוח בודק אם הערוצים נבדלים בציון, ומראה שההבדלים ביניהם אינם ניתנים להבחנה מרעש.

</div>


In [13]:
COMPONENTS = ["revenue_component", "satisfaction_component", "tenure_component"]
sc["value_score"] = 1 + 99 * sc[COMPONENTS].mean(axis=1)

assert sc.value_score.between(1, 100).all()
assert len(sc) == GOLDEN["sellers_scored"]

# Rule: the sellers with no in-window activity are counted and marked wherever
# the score is shown. A flag in the file is how that rule survives into notebook 03.
sc["no_activity_in_window"] = (sc.revenue_window == 0) & (sc.reviews_n == 0)
assert int(sc.no_activity_in_window.sum()) == GOLDEN["no_activity_in_window"]

final_sellers = sc[[
    "seller_id", "origin", "channel", "won_day", "exposure_days",
    "revenue_window", "revenue_outside_window", "revenue_all_time",
    "revenue_log", "revenue_component",
    "reviews_n", "review_mean", "review_shrunk", "satisfaction_component",
    "active_span_days", "n_orders", "first_order_day", "tenure_exposure_days",
    "tenure_ratio", "tenure_component",
    "no_activity_in_window", "value_score",
]].sort_values("value_score", ascending=False)

final_sellers.to_csv(CLEAN / "final_sellers.csv", index=False)
print("wrote final_sellers.csv :", final_sellers.shape)
print("value_score  min %.1f  median %.1f  max %.1f"
      % (sc.value_score.min(), sc.value_score.median(), sc.value_score.max()))
display(final_sellers.head())


wrote final_sellers.csv : (341, 22)
value_score  min 21.4  median 62.7  max 90.3


,seller_id,origin,channel,won_day,exposure_days,revenue_window,revenue_outside_window,revenue_all_time,revenue_log,revenue_component,reviews_n,review_mean,review_shrunk,satisfaction_component,active_span_days,n_orders,first_order_day,tenure_exposure_days,tenure_ratio,tenure_component,no_activity_in_window,value_score
240,ba90964cff9b9e0e6f32b23b82465f7b,organic_search,organic_search,2018-02-09,201,28265.80,15946.70,44212.50,4.451277,0.912630,25,4.560000,4.504909,0.868463,171,38,2018-02-25,185,0.924324,0.924324,False,90.278791
99,516e7738bd8f735ac19a010ee5450d8d,paid_search,paid_search,2018-05-08,113,6078.59,384.60,6463.19,3.783874,0.775795,68,4.661765,4.632155,0.955730,105,74,2018-05-11,110,0.954545,0.954545,False,89.640326
261,c70c1b0d8ca86052f45a432a38b73958,paid_search,paid_search,2018-03-13,169,15602.59,20686.56,36289.15,4.193225,0.859723,107,4.401869,4.394172,0.792518,145,260,2018-04-06,145,1.000000,1.000000,False,88.523944
284,d566c37fa119d5e66c4e9052e83ee4ea,organic_search,organic_search,2018-02-28,182,2608.90,2536.80,5145.70,3.416624,0.700499,37,4.702703,4.646364,0.965475,163,65,2018-03-16,166,0.981928,0.981928,False,88.380753
219,a63bfbaa882c8f4542891b4e2246cc7f,referral,referral,2018-05-24,97,5134.00,548.00,5682.00,3.710540,0.760759,5,5.000000,4.614728,0.943779,73,6,2018-06-11,79,0.924051,0.924051,False,87.743440


<div dir="rtl">

## השלמת טבלת הערוצים

נוספים ממוצע הציון, סטיית התקן, מספר המוכרים שקיבלו ציון ורווח סמך לממוצע. הרווח מחושב במכפיל t ולא במכפיל נורמלי, משום שערוץ ההפניות מחזיק תשעה מוכרים בלבד ובגודל כזה המכפיל הנורמלי צר מדי. ארבעת המוכרים שמקורם ריק אינם משתייכים לאף ערוץ ואינם נספרים כאן.

הערכים האלה נועדו להצגה עם רווחי הסמך שלהם. **אין להציג אותם כדירוג**, משום שהרווחים חופפים.

</div>


In [14]:
g = sc.dropna(subset=["channel"]).groupby("channel").value_score

score_stats = pd.DataFrame({
    "sellers_scored": g.size(),
    "score_mean"    : g.mean(),
    "score_std"     : g.std(),
}).reset_index()
# t, not z: the smallest channel holds nine sellers, where 1.96 is too narrow.
tcrit = score_stats.sellers_scored.map(lambda n: stats.t.ppf(0.975, n - 1) if n > 1 else np.nan)
score_stats["score_ci_half"] = tcrit * score_stats.score_std / np.sqrt(score_stats.sellers_scored)
score_stats["score_ci_low"]  = score_stats.score_mean - score_stats.score_ci_half
score_stats["score_ci_high"] = score_stats.score_mean + score_stats.score_ci_half

final_channels = ch.merge(score_stats.drop(columns="score_ci_half"), on="channel", how="left")
# other_publicities has no scored sellers. That is a real zero and a finding, so it
# is written as 0 rather than left as the join's absence marker.
final_channels["sellers_scored"] = final_channels.sellers_scored.fillna(0).astype(int)
final_channels.to_csv(CLEAN / "final_channels.csv", index=False)

results["largest_channel_sellers"] = int(score_stats.sellers_scored.max())
print("wrote final_channels.csv :", final_channels.shape)
print("largest channel, scored sellers:", results["largest_channel_sellers"])
assert results["largest_channel_sellers"] == GOLDEN["largest_channel_sellers"]
display(final_channels)


wrote final_channels.csv : (8, 17)
largest channel, scored sellers: 102


,channel,deals_all,activated_all_deals,sellers_ever_sold,leads,deals_eligible,activated_eligible,conversion_rate,activation_rate,activation_ci_low,activation_ci_high,excluded_from_ranking,sellers_scored,score_mean,score_std,score_ci_low,score_ci_high
0,paid_search,195,94,101,1586,146,81,0.122951,0.554795,0.473801,0.632979,False,87,59.422067,17.159952,55.764785,63.079349
1,direct_traffic,56,25,31,499,49,22,0.112224,0.448980,0.318526,0.586851,False,28,52.378996,18.447507,45.225800,59.532192
2,referral,24,8,9,284,18,8,0.084507,0.444444,0.245592,0.662839,False,9,64.848297,16.009499,52.542307,77.154288
3,social,75,29,31,1350,55,24,0.055556,0.436364,0.313732,0.567304,False,26,56.912775,16.631080,50.195336,63.630213
4,unknown,179,68,81,1099,153,63,0.162875,0.411765,0.336872,0.490980,True,75,59.171178,16.484538,55.378430,62.963926
5,organic_search,271,93,113,2296,207,84,0.118031,0.405797,0.341209,0.473818,False,102,59.005867,18.744947,55.324010,62.687724
6,small_channels_merged,25,7,10,761,23,7,0.032852,0.304348,0.156038,0.508661,True,10,51.812844,19.131061,38.127307,65.498381
7,other_publicities,3,0,0,65,3,0,0.046154,0.000000,0.000000,0.561506,True,0,NaN,NaN,NaN,NaN


<div dir="rtl">

## בדיקות קבלה

כל ערך בטבלה חושב מהנתונים הגולמיים **בנפרד** מהקוד של המחברת הזו. אי התאמה כאן משמעה שהצנרת מחשבת משהו אחר ממה שהוכרע, ולא שהיעד שגוי.

בסוף נבדקת שוב טביעת האצבע של הנתונים הגולמיים, כדי לוודא שהמחברת לא כתבה לשום קובץ מקור.

</div>


In [15]:
rows, ok = [], True
for k, expected in GOLDEN.items():
    got = results.get(k, "not computed")
    passed = (abs(got - expected) < 1e-9) if isinstance(got, float) else (got == expected)
    ok &= passed
    rows.append({"check": k, "expected": expected, "got": got,
                 "": "PASS" if passed else "FAIL"})
display(pd.DataFrame(rows).set_index("check"))

FINGERPRINT_AFTER = raw_fingerprint()
raw_intact = FINGERPRINT_AFTER == FINGERPRINT_BEFORE
print("\nraw data untouched :", raw_intact, "(", FINGERPRINT_AFTER[:16], "...)")
print("files written      :", sorted(p.name for p in CLEAN.glob("final_*.csv")))

assert raw_intact, "data/raw was modified, and this notebook must never write there"
assert ok, "one or more acceptance checks failed"
print("\nALL ACCEPTANCE CHECKS PASSED")


,expected,got,
check,,,
data_end,2018-08-29,2018-08-29,PASS
total_rows_11_tables,1559764,1559764,PASS
leads,8000,8000,PASS
deals,842,842,PASS
deals_eligible,665,665,PASS
activated_in_window,292,292,PASS
activation_pct,43.9,43.9,PASS
paid_search_activated,81,81,PASS
paid_search_deals,146,146,PASS



raw data untouched : True ( 42fca2bccb836569 ...)
files written      : ['final_channels.csv', 'final_deals.csv', 'final_sellers.csv']

ALL ACCEPTANCE CHECKS PASSED


<div dir="rtl">

## מה נבנה כאן, ומה הלאה

שלושה קבצים נכתבו לתיקייה `data/clean`, וכולם ניתנים לשחזור מלא מהנתונים הגולמיים ומהמחברת הזו:

| קובץ | שורות | תוכן |
|---|---|---|
| `final_deals.csv` | 842 | עסקה, ערוץ אחרי האיחוד, מועד חתימה, ימי חשיפה, כשירות לחלון, מועד המכירה הראשונה, הימים עד אליה והאם הופעלה |
| `final_sellers.csv` | 341 | שלושת הרכיבים וערכי הביניים שלהם, ההכנסה בחלון ומחוצה לו, שני מדדי הזמן בנפרד, דגל למוכרים ללא פעילות בחלון, הערוץ והציון הסופי |
| `final_channels.csv` | 8 | לידים, עסקאות ומוכרים בשתי האוכלוסיות בשמות נפרדים, שיעורי המרה והפעלה, רווחי סמך, סטטיסטיקת הציון ודגל להחרגה מדירוג |

**מה עוד לא נעשה.** לא הורץ שום מבחן מובהקות, לא נבנה שום תרשים ולא נקבעה שום מסקנה. מבחן התמורות, מבחני ההשוואה בין הערוצים, בדיקת הרגישות למשקלות והתרשימים נמצאים כולם במחברת 03.

</div>
